In [24]:
from spark_windows import configure_windows_hadoop

configure_windows_hadoop()


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    TimestampType,
)

from pyspark.sql import functions as F

In [26]:
spark = (
    SparkSession.builder.master("local[4]")
    .appName("Data Engineering")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [50]:
# "order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"

orders_schema = StructType(
    [
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True),
    ]
)

orders_schema

StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

In [51]:
df_orders = spark.read.csv(
    "./data/raw/olist_orders_dataset.csv",
    header=True,
    schema=orders_schema,
    timestampFormat="yyyy-MM-dd HH:mm:ss",
)

In [52]:
df_orders.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [54]:
df_orders.createOrReplaceTempView("orders")

In [ ]:
# spark.sql(
#     # "SELECT count(*) FROM orders"
#     "SELECT * FROM orders where order_purchase_timestamp IS NULL OR order_delivered_customer_date IS NULL"
# ).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|136cce7faa42fdb2c...|ed0271e0b7da060a3...|    invoiced|     2017-04-11 12:22:08|2017-04-13 13:25:17|                        NULL|                         NULL|          2017-05-09 00:00:00|
|ee64d42b8cf066f35...|caded193e8e47b836...|     shipped|     2018-06-04 16:44:48|2018-06-05 04:31:18|         2018-06-05 14:32:00|                         NULL|          2018-06-28 00:00:00|
|0760a852e4e9d89eb...|d2a79636084590b74...|  

In [56]:
df_orders.write.mode("overwrite").parquet("./data/bronze/olist_orders_dataset.parquet")

In [57]:
def add_bronze_metadata(dataframe):
    return (
        dataframe
    )

other_tables = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
]

for table_name in other_tables:
    source_path = f"./data/raw/{table_name}.csv"
    df_table = spark.read.csv(
        source_path,
        header=True,
        inferSchema=True,
    )
    df_table_with_metadata = add_bronze_metadata(df_table)
    df_table_with_metadata.write.mode("overwrite").parquet(
        f"./data/bronze/{table_name}.parquet"
    )